# Сбор данных с сайта «Детский мир»

**Цель проекта:** собрать данные о товарах из каталога «Спорт и отдых» с помощью `requests` и `BeautifulSoup`, структурировать их в `pandas` и сохранить результат в CSV.

**Источник данных:** [каталог «Спорт и отдых»](https://www.detmir.ru/catalog/index/name/sport_otdyh/).

В итоговую таблицу входят название товара, текущая цена, старая цена (если указана), рейтинг, код товара и артикул.

## 1. Импорт библиотек

In [1]:
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

## 2. Настройки запроса и вспомогательные функции

In [2]:
BASE_URL = 'https://www.detmir.ru'
CATALOG_URL = f'{BASE_URL}/catalog/index/name/sport_otdyh/'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36'
}


def get_soup(url: str) -> BeautifulSoup:
    """Загружает страницу и возвращает объект BeautifulSoup."""
    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'lxml')


def safe_get_text(element, default=None):
    """Безопасно извлекает текст из HTML-элемента."""
    return element.get_text(' ', strip=True) if element else default


def price_to_number(value):
    """Преобразует цену с сайта в число; пропуски оставляет как None."""
    if value is None:
        return None
    digits = re.sub(r'\D', '', value)
    return float(digits) if digits else None

## 3. Извлечение кода товара и артикула со страницы товара

На странице товара эти значения находятся в таблице характеристик. Название товара отдельно не сохраняем: оно уже извлекается из карточки каталога.

In [3]:
def get_product_identifiers(url: str) -> dict:
    """Возвращает код товара и артикул со страницы конкретного товара."""
    soup = get_soup(url)

    result = {
        'item_code': None,
        'artikul': None,
    }

    for row in soup.select('tr[data-testid="characteristicRow"]'):
        cells = row.find_all(['th', 'td'])
        if len(cells) != 2:
            continue

        key = safe_get_text(cells[0], '')
        value = safe_get_text(cells[1])

        if 'Код товара' in key:
            result['item_code'] = value
        elif 'Артикул' in key:
            result['artikul'] = value

    return result

## 4. Сбор данных из каталога

Собираем товары с первых трёх страниц каталога. Для каждой карточки извлекаем название, рейтинг и цены, затем открываем страницу товара, чтобы получить код и артикул.

In [4]:
page_items = []

for page in range(1, 4):
    page_url = f'{CATALOG_URL}?page={page}'
    soup = get_soup(page_url)
    product_cards = soup.select('section[id*="product"]')

    for card in product_cards:
        title_link = card.select_one('a[data-testid="titleLink"]')
        if title_link is None:
            continue

        name = safe_get_text(title_link.select_one('span'))
        product_url = title_link.get('href')
        if not product_url:
            continue
        if product_url.startswith('/'):
            product_url = BASE_URL + product_url

        rating_elements = card.select(
            '[data-testid="rating"] [data-testid="typography"]'
        )
        rating = safe_get_text(rating_elements[-1]) if rating_elements else None

        current_price_text = None
        old_price_text = None
        price_block = card.select_one('[data-testid="productPrice"]')

        if price_block:
            current_price = price_block.select('span.diTAa')
            old_price = price_block.select_one('span.iwVsa')

            if current_price:
                current_price_text = safe_get_text(current_price[0])
            old_price_text = safe_get_text(old_price)

        identifiers = get_product_identifiers(product_url)

        page_items.append({
            'name': name,
            'price': price_to_number(current_price_text),
            'old_price': price_to_number(old_price_text),
            'rating': rating,
            'item_code': identifiers['item_code'],
            'artikul': identifiers['artikul'],
        })

    print(f'Страница {page}: найдено карточек — {len(product_cards)}')
    time.sleep(1)

Страница 1: найдено карточек — 33
Страница 2: найдено карточек — 36
Страница 3: найдено карточек — 36


## 5. Формирование таблицы и первичная проверка

In [5]:
df = pd.DataFrame(page_items)

print('Размер таблицы:', df.shape)
display(df.head())

Размер таблицы: (105, 6)


,name,price,old_price,rating,item_code,artikul
0,Санки Happy Baby Черный,4498.0,5055.0,"4,9",1000346909,51001-ч
1,Тюбинг F78 100 см,3483.0,4466.0,5,1001008060,1.00-ОК-01-ГУСИ
2,Коньки Calambus Раздвижные 24-34,1509.0,2253.0,"4,9",1000874815,ЦБ-00029591
3,Снегокат NiKA kids,6318.0,8100.0,5,1000241945,ЦБ-00024552
4,Тюбинг ТЯНИТОЛКАЙ 83 см,1663.0,2772.0,None,1001116050,TSRD00151044


In [6]:
print('Типы данных:')
print(df.dtypes)

print('\nКоличество пропусков:')
print(df.isna().sum())

print('\nКоличество дубликатов по коду товара:')
print(df.duplicated(subset='item_code').sum())

Типы данных:
name          object
price        float64
old_price    float64
rating        object
item_code     object
artikul       object
dtype: object

Количество пропусков:
name          0
price         0
old_price    40
rating        8
item_code     0
artikul       0
dtype: int64

Количество дубликатов по коду товара:
3


## 6. Проверка цен и товаров со скидкой

`price` — текущая цена, `old_price` — прежняя цена, если она отображается на сайте. Отсутствие `old_price` обычно означает, что старая цена в карточке не указана.

In [7]:
display(df[['name', 'price', 'old_price']].head(10))

products_with_old_price = df[df['old_price'].notna()]
display(products_with_old_price[['name', 'price', 'old_price']].head(10))

,name,price,old_price
0,Санки Happy Baby Черный,4498.0,5055.0
1,Тюбинг F78 100 см,3483.0,4466.0
2,Коньки Calambus Раздвижные 24-34,1509.0,2253.0
3,Снегокат NiKA kids,6318.0,8100.0
4,Тюбинг ТЯНИТОЛКАЙ 83 см,1663.0,2772.0
5,Качалка UNIX Kids Лошадка серый,6594.0,9842.0
6,Каталка BabyGo Гусеница желтый,4499.0,4999.0
7,Матрас для санок BabyGo Серый,1499.0,NaN
8,Снегокат NiKA kids ТС6/П с пингвинятами,7620.0,12700.0
9,Снегокат NiKA kids Тимка спорт 4-1,4489.0,5299.0


,name,price,old_price
0,Санки Happy Baby Черный,4498.0,5055.0
1,Тюбинг F78 100 см,3483.0,4466.0
2,Коньки Calambus Раздвижные 24-34,1509.0,2253.0
3,Снегокат NiKA kids,6318.0,8100.0
4,Тюбинг ТЯНИТОЛКАЙ 83 см,1663.0,2772.0
5,Качалка UNIX Kids Лошадка серый,6594.0,9842.0
6,Каталка BabyGo Гусеница желтый,4499.0,4999.0
8,Снегокат NiKA kids ТС6/П с пингвинятами,7620.0,12700.0
9,Снегокат NiKA kids Тимка спорт 4-1,4489.0,5299.0
13,Снегокат NiKA kids Snowpatrol,6999.0,11999.0


## 7. Сохранение результата

In [8]:
df.to_csv('detmir_products.csv', index=False, encoding='utf-8-sig')
print('Файл detmir_products.csv сохранён')

Файл detmir_products.csv сохранён


## 8. Итоги

В результате сформирована таблица с данными о товарах каталога «Спорт и отдых». Перед использованием результатов стоит учитывать, что структура HTML сайта может меняться, поэтому CSS-селекторы со временем могут потребовать обновления.